# Study 950 — Zero-Coupon Convexity 📐

**Does a zero-coupon Treasury fund pay you for convexity, or just for duration?**

A bond's price curves against its yield. That curvature — *convexity* — is supposed to be a
gift: for the same duration, a more convex position gains more on a big rally than it loses
on an equally big sell-off. The zero-coupon Treasury funds (**EDV**, 20-30y STRIPS;
**ZROZ**, 25y+) sit further out the curve than the ordinary long-bond fund **TLT**, so per
dollar of duration they carry more curvature. If that curvature is being *paid*, the zero
should beat a **duration-matched** TLT + T-bill mix in **large-move months** and lose a
little in quiet ones.

We match the duration on the realised beta of each leg to **the same rate factor** (the
daily change in the 30-year yield, `^TYX`), race the two arms **excess-of-cash**, and test
the **asymmetry explicitly** — a regression on the squared rate move — rather than reading
it off an average. EDV vs the matched mix, 2009-02-02 → 2026-06-30 (4,377 days,
208 months).

*Every real number below is the frozen headline (`docs/results.md`, Fingerprint
`a669055b6e7a`); the only live cells run the fast offline synthetic control. As-of 2026-06-30.*


## 1. What convexity is, in one breath

If rates move 1%, a 20-year bond loses about 20% — *roughly*. The word doing the work is *roughly*: the loss on a rise is slightly smaller than the gain on a fall of the same size, because the price-yield line bends. That bend is convexity. The bigger the move, the more the bend matters — which is why any real test has to look at the **size** of the move, not the average month.

> 🔬 **For the quants** — formally `ΔP/P ≈ −D·Δy + ½·C·Δy²`. The first term is duration and it is what we neutralise; the whole study lives in the second term.

## 2. First, make the two arms carry the same rate risk

The zero fund is simply *longer*. Solving the match from realised sensitivity to the 30-year yield says you need **1.42 units of TLT** (funded with T-bills) to carry as much rate risk as one unit of EDV — the STRIPS fund runs about **42% more duration per dollar**. Do that, and the two arms end up with volatilities that agree to three decimal places.

In [1]:
R = dict(L_mean=1.42, a_vol=21.41, b_vol=21.42, vol_ratio=1.0, sp_mean=-0.53, sp_bp_mo=-3.98,
         sp_bp_t=-1.08, sp_hit=44.2)
print('one unit of EDV  ~ %.2f units of TLT (rest in T-bills)' % R['L_mean'])
print('volatility  EDV arm %.2f%%   matched mix %.2f%%   ratio %.3f'
      % (R['a_vol'], R['b_vol'], R['vol_ratio']))
print('the leftover spread: %+.2f%%/yr  = %+.2f bp per month (t %+.2f), '
      'positive in only %.1f%% of months'
      % (R['sp_mean'], R['sp_bp_mo'], R['sp_bp_t'], R['sp_hit']))

one unit of EDV  ~ 1.42 units of TLT (rest in T-bills)
volatility  EDV arm 21.41%   matched mix 21.42%   ratio 1.000
the leftover spread: -0.53%/yr  = -3.98 bp per month (t -1.08), positive in only 44.2% of months


## 3. The honest headline — the average says nothing

Over seventeen and a half years the zero fund ends up **-0.53%/yr** behind the duration-matched mix — with a *t* of -0.87 and a bootstrap range of [-11.0, +3.4] bp/month. That is a shrug, not a result. But an average was never the right test: convexity is supposed to pay in the **big** months and cost you in the quiet ones.

## 4. So: does it pay in the big months?

Sort the months into three buckets by how far the 30-year yield actually moved. If the convexity story were true, the numbers should climb from left to right.

In [2]:
rows = [('quiet', 70, 4.1, -0.94, -0.15), ('middling', 69, 14.5, -7.51, -1.22),
        ('large', 69, 32.4, -3.52, -0.43)]
print('bucket      n   mean |move|   spread      t     (after removing the\n                                                        residual duration leak)')
for name, n, dy, sp, t in rows:
    print('%-9s %3d   %6.1f bp   %+7.2f bp  %+5.2f' % (name, n, dy, sp, t))
print()
print('hedged: quiet %+.2f   middling %+.2f   large %+.2f  bp/month'
      % (-1.22, -7.07, -2.38))

bucket      n   mean |move|   spread      t     (after removing the
                                                        residual duration leak)
quiet      70      4.1 bp     -0.94 bp  -0.15
middling   69     14.5 bp     -7.51 bp  -1.22
large      69     32.4 bp     -3.52 bp  -0.43

hedged: quiet -1.22   middling -7.07   large -2.38  bp/month


They do not climb. The large-move bucket is **-3.52 bp/month** — negative — and it stays negative after we strip out the small amount of leftover duration the match could not remove. On the rawest possible reading, the convexity paycheck simply does not show up.

## 5. The regression is kinder — and still not enough

Fit the shape properly (`spread = a + b1·move + b2·move²`) and the signs come out **exactly as the textbook says they should**: a positive curvature term (**+110.5**, worth about **+6.9 bp** in a 25 bp month and **+27.6 bp** in a 50 bp one) and a negative intercept (**-8.75 bp/month** — the price you pay for it). That pattern repeats in **11 of the 12** cuts we ran (two funds × three eras × two ways of measuring the move — the whole census, printed in [docs/results.md](../docs/results.md)).

And yet the *t*-statistic on that curvature term is **+0.84**. The desk's bar is |*t*| ≥ 2, and exactly **2 of 12** cuts reach it — both of them the *same* 84 months of the cross-check fund ZROZ (2010-2017: **+2.14** and **+2.88**), counted once per regressor. What happens next door settles it: on **2018-2026** that same fund gives a curvature term of **-123.5** with a *positive* intercept — the story running backwards — and EDV, the fund in the headline, never gets past **+1.87** in any era. A shape that reaches significance in one window and inverts in the one beside it is a coin, not a premium.

## 6. The number that settles it — a 28 bp breakeven

Take the fit at face value and ask the practical question: **how far must rates move in a month before the curvature gain repays the carry you gave up?** The answer is **28 basis points**. The *median* month in this sample moved **15 bp**. So even on its own most flattering terms, the convexity trade is under water in roughly two months out of three, and needs the tail to bail it out.

## 7. One more inconvenient detail

EDV holds 20-30 year STRIPS; TLT holds 20-year-plus coupon bonds. They are not at the same point on the curve, and the match — solved against a single 30-year factor — cannot fully fix that. What is left is about **+0.72 years** of residual duration (*t* = -2.73), and it is there in **both** halves of the sample. In other words, a good part of what this "convexity spread" trades is the 20s-versus-30s curve. That is a legitimate position — it is just not the one the story is selling.

## 8. Is the tool broken, or is the effect small? (live, offline)

Fair question. So we build a synthetic bond world where the long leg *genuinely* carries a big convexity pickup and genuinely pays carry for it, and a null world where it does not — and run the identical machinery on both.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from zero_convexity import data, strategy as st
planted = [st.synthetic_detect(data.synthetic_panel(signal_strength=1.0, seed=950+s)[0])
           for s in range(6)]
null    = [st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=950+s)[0])
           for s in range(6)]
def show(tag, runs):
    b2 = np.array([r['b2'] for r in runs]); t = np.array([r['b2_t'] for r in runs])
    a  = np.array([r['a_bp_mo'] for r in runs])
    print('%-14s curvature term %+8.1f  (t %+5.2f)  price of it %+7.2f bp/mo  '
          '[6 worlds, |t|>=2 in %d]'
          % (tag, b2.mean(), t.mean(), a.mean(), (abs(t) >= 2).sum()))
show('planted world', planted)
show('null world', null)

planted world  curvature term   +324.2  (t +6.69)  price of it  -28.59 bp/mo  [6 worlds, |t|>=2 in 6]
null world     curvature term    -37.1  (t -0.91)  price of it   +2.00 bp/mo  [6 worlds, |t|>=2 in 1]


The detector fires hard when there is something to find and goes quiet when there is not. So the real-tape silence is a fact about the **Treasury curve**, not about the harness.

## Verdict

- **Signal — Weak.** The shape is right almost everywhere we look — positive curvature term and negative intercept in **11 of 12** cuts — but the headline *t* is **+0.84**, the bootstrap interval straddles zero, and the large-move bucket is negative. The **2 cuts that do clear 2** are one window of the cross-check fund whose neighbouring window flips both signs, so nothing here replicates; on top of that a significant **+0.72 yr** of residual duration means part of the spread is a curve trade. Directionally right, statistically uncertified.
- **Tradability — Mirage.** The spread earns **-0.53%/yr** and needs a **28 bp** month just to break even against a median month of **15 bp**. Costs and financing never change the sign — this one fails at the signal stage, not the friction stage.
- **What the zero fund *is* good for:** more duration per dollar (about 42% more). That is a real, useful property. It is just not a convexity paycheck.